Hi there!
This is the code template for CW2 task1 of COMP34711 2025/26.

- <span style="color:red; font-size:1em">First of all, please rename the notebook into "{your_student_id}_CW2_task{your_task_number}.ipynb", for example "12345678_CW2_task1.ipynb".</span>

- In this template, we only provide the minimal structure for your coursework.
  
- Please carefully read and organize your code in the template we provided.

## Constants

In [1]:
#Please keep only necessary information in this cell.

#----------------------Please keep all following constants unchanged.----------------------------------------
NUM_ROWS_VALIDATION = 1031 # Number of rows in validation set
NUM_ROWS_TEST = 1053 # Number of rows in test set

#----------------------Please modify the following constants to fit your actual value.-----------------------
STUDENT_ID = '10879360'  # Replace with your actual 8-digits student ID
TRAINING_SET = './data/CW2_training_dataset.csv' # Replace with the actual path to your training dataset csv file
VALIDATION_SET = './data/CW2_validation_dataset.csv'  # Replace with the actual path to your validation dataset csv file
VALIDATION_SET_OUTPUT = f'./data/{STUDENT_ID}_CW2_task1_validation_results.csv'  # Replace with the actual path to your validation prediction csv file
TEST_SET_INPUT = './data/CW2_test_dataset.csv'  # Replace with the actual path to your test prediction csv file

#----------------------Your constants------------------------------------------------
# By adding more constants here, you can help improve the clarity and maintainability of your code and make the reviewing easier for TAs.

## Installations

In [2]:
# Download pre-trained data for embedding, will use glove.6B.300d.txt only.
!wget -P . http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip
GLOVE_PATH = './glove.6B.300d.txt'

--2025-12-02 15:06:11--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-12-02 15:06:11--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-12-02 15:06:11--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘./glove.6B.zip’

gl

In [3]:
# Install required packages for the coursework
# Uncomment and run the following lines if needed

!pip install pandas scikit-learn torch nltk --quiet

## Imports

In [4]:
#Please keep all imports of your code cells in this cell

#---------------------Required imports----------------------
import pandas as pd
import re
import sys
import os.path
import csv
from sklearn.metrics import f1_score
#----------------------Your imports-------------------------
import numpy as np

# Pytorch tools
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# NLTK tools
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from collections import Counter

# Fix seed for stable result
def set_seed(seed_value=378):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
set_seed(378)

# Check a device that will run this code below
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Using device: cuda


## Start of your code cells

- The code cells provided below are demo code format for TAs to quickly locate your implementation.

- You have full right to freely add/delete/edit the titles and codes in the following cells.

- Please follow this genre order: "comedy, cult, flashback, historical, revenge, romantic, scifi, violence".

### Data Loading

In [5]:
# Read data from given csv files
df_train = pd.read_csv(TRAINING_SET)
df_validation = pd.read_csv(VALIDATION_SET)

# Check content of files
print(df_train.head())
print("\n----- Data Shapes -----")
print(f"Training set shape: {df_train.shape}")
print(f"Validation set shape: {df_validation.shape}")

                                     ID                        title  \
0  ee7722b2-bc23-400b-9461-4ff91f01f486                  Next of Kin   
1  3b111f7d-0c19-4cb3-84a1-d6dc687c9716              The Survivalist   
2  3116352f-4b50-43a2-b9be-458c4aa086e5                  Superman II   
3  bbb71d71-1503-4aa6-9129-918b9efc3c3f  The Hunchback of Notre Dame   
4  c12f67ca-5825-43d0-b9a7-61c348915715                         Taxi   

                                       plot_synopsis  comedy  cult  flashback  \
0  Truman Gates (Patrick Swayze), raised in Appal...       0     0          0   
1  The film takes place when oil production has c...       0     1          0   
2  Before the destruction of Krypton, the crimina...       0     0          0   
3  The gypsy Esmeralda captures the hearts of man...       0     0          0   
4  Taxi portrays director Jafar Panahi as he cour...       0     0          0   

   historical  revenge  romantic  scifi  violence  
0           0        1      

### Tokenization

In [6]:
CONTRACTIONS = {
    "n't": " not", "'s": " is", "'re": " are", "'ll": " will",
    "'d": " would", "'ve": " have", "'m": " am"
}

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    # Clean text: lowercase, expand contractions, remove special characters.
    text = text.lower()

    # 1. Expand contractions (can't -> cannot)
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)

    # 2. Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+', '', text)

    # 3. Keep only alphanumeric characters and spaces
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text

# --------------------------------------------------------------------------------
# 1. Vocabulary Class (Lemmatization + Stopwords + Filtering)
# --------------------------------------------------------------------------------
class Vocabulary:
    def __init__(self, freq_threshold=3):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_eng(text):
        # 1. Clean text
        text = clean_text(text)
        tokens = word_tokenize(text)

        # 2. Remove stopwords and Lemmatize
        processed_tokens = []
        # Filter by length and stopwords
        for word in tokens:
            if len(word) >= 3 and word not in STOP_WORDS:
                lemma = LEMMATIZER.lemmatize(word) # Lemmatize to base form
                processed_tokens.append(lemma)

        return processed_tokens

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4 # Start index from 4 (0:PAD, 1:SOS, 2:EOS, 3:UNK)

        print("Building vocabulary with Lemmatization~")
        # 1. Tokenize sentences and count word frequencies
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence): # Custom tokenizer (includes Lemmatization/Stopword removal)
                frequencies[word] += 1

        # 2. Filter by frequency threshold and assign integer IDs
        for word, count in frequencies.items():
            if count >= self.freq_threshold: # Filter out rare words
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
        print(f"Vocabulary size after filtering: {len(self.stoi)}")

    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

# --------------------------------------------------------------------------------
# 2. Pre-processing Function (Head + Tail Truncation Strategy)
# --------------------------------------------------------------------------------
def preprocess_dataset(df, vocab, max_len=1400):
    # Tokenize data and apply truncation/padding.
    print(f"Processing {len(df)} samples~ (Head+Tail Truncation Strategy)")

    texts = df['title'] + " " + df['plot_synopsis']

    # Handle labels
    if len(df.columns) > 3:
        labels = df.iloc[:, 3:].values
    else:
        labels = np.zeros((len(df), 8))

    processed_X = []

    for text in texts:
        # 1. Numericalize
        numericalized = vocab.numericalize(text)

        # 2. Truncation
        # Calculate target length excluding SOS/EOS
        target_len = max_len - 2

        if len(numericalized) > target_len:
            # Truncate sequence if it exceeds max length
            numericalized = numericalized[:target_len]

        # 3. Add SOS / EOS tokens
        numericalized = [vocab.stoi["<SOS>"]] + numericalized + [vocab.stoi["<EOS>"]]

        # 4. Padding
        padding_len = max_len - len(numericalized)
        if padding_len > 0:
            numericalized += [vocab.stoi["<PAD>"]] * padding_len

        processed_X.append(numericalized)

    # Convert to tensors
    X_tensor = torch.tensor(processed_X, dtype=torch.long)
    y_tensor = torch.tensor(labels, dtype=torch.float)

    return X_tensor, y_tensor

# --------------------------------------------------------------------------------
# 3. Dataset Class & Execution
# --------------------------------------------------------------------------------
class MovieDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self): return len(self.X)
    def __getitem__(self, index): return self.X[index], self.y[index]

# Configuration (64 for Tesla 4 GPU)
BATCH_SIZE = 64
MAX_LEN = 1400

print("Building Optimized Vocabulary~")

# 1. Build Vocabulary
vocab = Vocabulary(freq_threshold=3)
vocab.build_vocabulary(df_train['title'] + " " + df_train['plot_synopsis'])
print(f"✅ Vocabulary built! Size: {len(vocab)} words (Stopwords removed + Lemmatized)")

# 2. Preprocessing
print("\nConverting text to tensors (Head+Tail Truncation)~")
X_train, y_train = preprocess_dataset(df_train, vocab, MAX_LEN)
X_val, y_val = preprocess_dataset(df_validation, vocab, MAX_LEN)

# 3. Create Dataset and DataLoaders
train_dataset = MovieDataset(X_train, y_train)
val_dataset = MovieDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"✅ DataLoaders ready. Batch size: {BATCH_SIZE}")

Building Optimized Vocabulary~
Building vocabulary with Lemmatization~
Vocabulary size after filtering: 44661
✅ Vocabulary built! Size: 44661 words (Stopwords removed + Lemmatized)

Converting text to tensors (Head+Tail Truncation)~
Processing 7127 samples~ (Head+Tail Truncation Strategy)
Processing 1031 samples~ (Head+Tail Truncation Strategy)
✅ DataLoaders ready. Batch size: 64


### Model and Training

In [7]:
# --------------------------------------------------------------------------------
# Load GloVe Embeddings Function
# --------------------------------------------------------------------------------
def load_glove_embeddings(glove_path, word2idx, embedding_dim=300):
    print(f"Loading GloVe embeddings from {glove_path}~")

    # 1. Load GloVe vectors into a dictionary
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector

    print(f"Loaded {len(glove_dict)} word vectors from GloVe.")

    # 2. Create embedding matrix corresponding to our vocabulary
    vocab_size = len(word2idx)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    found_count = 0
    for word, idx in word2idx.items():
        # Use GloVe vector if available
        if word in glove_dict:
            embedding_matrix[idx] = glove_dict[word]
            found_count += 1
        else:
            # Initialize OOV (Out-Of-Vocabulary) words with random normal noise
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

    # Explicitly set <PAD> vector to zeros
    if "<PAD>" in word2idx:
        embedding_matrix[word2idx["<PAD>"]] = np.zeros((embedding_dim,))

    print(f"✅ Embedding Matrix Created: {found_count}/{vocab_size} words found in GloVe.")

    return torch.tensor(embedding_matrix, dtype=torch.float)

In [8]:
# --------------------------------------------------------------------------------
# 1. Asymmetric Loss Class (ASL)
# --------------------------------------------------------------------------------
class AsymmetricLoss(nn.Module):
    # Asymmetric Loss for multi-label classification to handle class imbalance.
    # Down-weights easy negatives and hard-mines difficult samples.
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, disable_torch_grad=True):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.disable_torch_grad = disable_torch_grad

    def forward(self, x, y):
        targets = y
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos

        # Margin clipping for negatives
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        # Basic Cross Entropy components
        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))

        # Asymmetric Focusing Mechanism
        pos_weight = targets * (1.0 - xs_pos).pow(self.gamma_pos)
        neg_weight = (1.0 - targets) * xs_pos.pow(self.gamma_neg)

        loss = pos_weight * los_pos + neg_weight * los_neg

        return -loss.mean()

# --------------------------------------------------------------------------------
# 2. BiLSTMAttention Model (with GloVe support)
# --------------------------------------------------------------------------------
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout, pretrained_embedding=None):
        super(BiLSTMAttention, self).__init__()

        # Load pretrained embeddings if provided, otherwise initialize randomly
        if pretrained_embedding is not None:
            # freeze=False allows fine-tuning of GloVe vectors during training
            self.embedding = nn.Embedding.from_pretrained(pretrained_embedding, freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Bidirectional LSTM Layer
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )

        # Attention Mechanism
        self.attention = nn.Linear(hidden_dim * 2, 1)

        # Fully Connected Output Layer
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # 1. Embedding
        embedded = self.dropout(self.embedding(text))
        # 2. LSTM
        lstm_output, _ = self.lstm(embedded)
        # 3. Attention
        energy = self.attention(lstm_output)
        attn_weights = F.softmax(energy, dim=1)
        context_vector = torch.sum(attn_weights * lstm_output, dim=1)
        # 4. Classification
        output = self.fc(self.dropout(context_vector))
        return output

# --------------------------------------------------------------------------------
# 3. Training & Evaluation Functions
# --------------------------------------------------------------------------------

def train_epoch(model, iterator, optimizer, criterion, device):
    # Train the model for one epoch.
    model.train()
    epoch_loss = 0
    for texts, labels in iterator:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(texts)
        loss = criterion(predictions, labels)
        loss.backward()
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion, device, threshold=0.45):
    # Evaluate model using a fixed threshold.
    model.eval()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for texts, labels in iterator:
            texts, labels = texts.to(device), labels.to(device)
            predictions = model(texts)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()
            probs = torch.sigmoid(predictions)
            preds = (probs > threshold).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate Weighted F1 Score
    f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    return epoch_loss / len(iterator), 0, f1_weighted

def find_best_threshold_classwise(model, iterator, device):
    # Find the optimal probability threshold for each of the 8 classes individually.
    model.eval()
    all_probs = []
    all_labels = []

    # 1. Collect predictions and labels for the entire dataset
    with torch.no_grad():
        for texts, labels in iterator:
            texts = texts.to(device)
            predictions = model(texts)
            probs = torch.sigmoid(predictions)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    best_thresholds = [] # Store optimal threshold for each class

    # 2. Iterate through each of the 8 classes
    for i in range(8):
        best_f1 = 0.0
        best_thresh = 0.5

        y_true = all_labels[:, i]
        y_prob = all_probs[:, i]

        # Search for best threshold in range 0.30 to 0.70
        for thresh in np.arange(0.30, 0.71, 0.01):
            y_pred = (y_prob > thresh).astype(float)

            current_f1 = f1_score(y_true, y_pred, zero_division=0)

            if current_f1 > best_f1:
                best_f1 = current_f1
                best_thresh = thresh

        best_thresholds.append(best_thresh)

    # 3. Calculate final Weighted F1 using the optimized thresholds
    final_preds = np.zeros_like(all_probs)
    for i in range(8):
        final_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)

    total_weighted_f1 = f1_score(all_labels, final_preds, average='weighted', zero_division=0)

    return best_thresholds, total_weighted_f1

# --------------------------------------------------------------------------------
# 4. Main Execution Block
# --------------------------------------------------------------------------------

# 1. Hyperparameters
INPUT_DIM = len(vocab)
EMBEDDING_DIM = 300
HIDDEN_DIM = 512
OUTPUT_DIM = 8
N_LAYERS = 2
DROPOUT = 0.3
LEARNING_RATE = 1e-3
N_EPOCHS = 10

# Load GloVe embeddings if available
if os.path.exists(GLOVE_PATH):
    glove_matrix = load_glove_embeddings(GLOVE_PATH, vocab.stoi, EMBEDDING_DIM)
else:
    print(f"⚠️ Warning: GloVe file not found at {GLOVE_PATH}. Training from scratch.")
    glove_matrix = None

# 2. Initialize Model
print("\nInitializing Model (Bi-LSTM + Attention + ASL Loss)~")
model = BiLSTMAttention(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT, pretrained_embedding=glove_matrix)
model = model.to(device)

# 3. Loss & Optimizer
criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

print(f'The model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

# 4. Training Loop
best_valid_f1 = 0.0
best_thresholds_list = [0.5] * 8

print(f"\nStarting Training with Class-wise Threshold Optimization~")
print("-" * 115)
print(f"{'Epoch':^6} | {'Train Loss':^10} | {'Val Loss':^10} | {'Val F1(Fixed)':^13} | {'Best F1(Opt)':^13} | {'Result'}")
print("-" * 115)

for epoch in range(N_EPOCHS):
    # 1. Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)

    # 2. Evaluate (Fixed Threshold for reference)
    valid_loss, _, valid_f1_fixed = evaluate(model, val_loader, criterion, device, threshold=0.5)

    # 3. Optimize Thresholds (Class-wise)
    epoch_best_thresholds, epoch_best_f1 = find_best_threshold_classwise(model, val_loader, device)

    # 4. Update Scheduler
    scheduler.step(epoch_best_f1)

    # 5. Save Model (based on optimized F1)
    if epoch_best_f1 > best_valid_f1:
        best_valid_f1 = epoch_best_f1
        best_thresholds_list = epoch_best_thresholds # Save the list of optimal thresholds
        torch.save(model.state_dict(), f'{STUDENT_ID}_task1_best_model.pt')
        saved_msg = "🔥 Saved"
    else:
        saved_msg = ""

    # 6. Print Progress
    print(f"{epoch+1:02}/{N_EPOCHS:02}  | {train_loss:.4f}     | {valid_loss:.4f}     | {valid_f1_fixed:.4f}        | {epoch_best_f1:.4f}        | {saved_msg}")

print("-" * 115)
print(f"✅ Training Complete. All-time Best Weighted F1: {best_valid_f1:.4f}")
print(f"📌 Final Best Thresholds: {best_thresholds_list}")

# 5. Final Validation Check
print("\n--- Final Check on Loaded Model ---")
try:
    model.load_state_dict(torch.load(f'{STUDENT_ID}_task1_best_model.pt'))
    model.to(device)
    final_threshs, final_f1 = find_best_threshold_classwise(model, val_loader, device)
    print(f"✅ Final Score -> F1: {final_f1:.4f}")
    print(f"✅ Thresholds: {final_threshs}")
except FileNotFoundError:
    print("Model file not found.")

Loading GloVe embeddings from ./glove.6B.300d.txt~
Loaded 400000 word vectors from GloVe.
✅ Embedding Matrix Created: 38294/44661 words found in GloVe.

Initializing Model (Bi-LSTM + Attention + ASL Loss)~
The model has 23,041,317 trainable parameters

Starting Training with Class-wise Threshold Optimization~
-------------------------------------------------------------------------------------------------------------------
Epoch  | Train Loss |  Val Loss  | Val F1(Fixed) | Best F1(Opt)  | Result
-------------------------------------------------------------------------------------------------------------------
01/10  | 0.0875     | 0.0824     | 0.5339        | 0.5572        | 🔥 Saved
02/10  | 0.0784     | 0.0788     | 0.5671        | 0.5802        | 🔥 Saved
03/10  | 0.0691     | 0.0824     | 0.5717        | 0.5845        | 🔥 Saved
04/10  | 0.0561     | 0.0877     | 0.5595        | 0.5760        | 
05/10  | 0.0401     | 0.1160     | 0.5432        | 0.5668        | 
06/10  | 0.0273     | 

### Prediction

In [14]:
# --------------------------------------------------------------------------------
# 1. Prediction Configuration
# --------------------------------------------------------------------------------
MODEL_PATH = f'{STUDENT_ID}_task1_best_model.pt'
OPTIMAL_THRESHOLDS = best_thresholds_list
MAX_LEN = 1400 # Must match the MAX_LEN used during training

# Model Hyperparameters (Must match the architecture of the saved model)
MODEL_CONFIG = {
    'max_len': MAX_LEN,
    'embedding_dim': 300,
    'hidden_dim': 512,
    'output_dim': 8,
    'n_layers': 2,
    'dropout': 0.3
}

# --------------------------------------------------------------------------------
# 2. Prediction Function
# --------------------------------------------------------------------------------
print("--- Starting Final Single Model Prediction ---")

# 1. Load Model & Configuration
print(f"Loading BEST Model from: {MODEL_PATH}")

# Initialize Model Architecture
try:
    model = BiLSTMAttention(
        vocab_size=len(vocab),
        embedding_dim=MODEL_CONFIG['embedding_dim'],
        hidden_dim=MODEL_CONFIG['hidden_dim'],
        output_dim=MODEL_CONFIG['output_dim'],
        n_layers=MODEL_CONFIG['n_layers'],
        dropout=MODEL_CONFIG['dropout']
    )
except NameError:
    print("\n❌ Error: BiLSTMAttention or Vocabulary class not defined. Run setup cells first.")

# Load trained weights (map_location handles CPU/GPU compatibility)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

# 2. Load & Preprocess Target Data
# Note: Replace VALIDATION_SET with TEST_SET_INPUT for the final submission.
df_target = pd.read_csv(TEST_SET_INPUT)

# Preprocessing
X_target, y_target_dummy = preprocess_dataset(df_target, vocab, MODEL_CONFIG['max_len'])

target_dataset = MovieDataset(X_target, y_target_dummy)
target_loader = DataLoader(target_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Prediction Loop (Calculate Probabilities)
all_probs = []
with torch.no_grad():
    for texts, _ in target_loader:
        texts = texts.to(device)
        predictions = model(texts)
        probs = torch.sigmoid(predictions)
        all_probs.extend(probs.cpu().numpy())

all_probs = np.array(all_probs)

# 4. Apply Class-wise Thresholds & Convert to Integers
print(f"Applying Class-wise thresholds: {OPTIMAL_THRESHOLDS}")

# Initialize integer array for binary predictions (0 or 1)
final_predictions_binary = np.zeros(all_probs.shape, dtype=int)

for i in range(8):
    # Apply specific threshold for each class i
    final_predictions_binary[:, i] = (all_probs[:, i] > OPTIMAL_THRESHOLDS[i]).astype(int)

# 5. Generate Submission CSV

# Extract IDs (First column of the target DataFrame)
test_ids = df_target.iloc[:, 0].values

# Create DataFrame for predictions
df_pred = pd.DataFrame(final_predictions_binary,
                          columns=['comedy', 'cult', 'flashback', 'historical',
                                  'revenge', 'romantic', 'scifi', 'violence'])

# Ensure integer data type
df_pred = df_pred.astype(int)

df_ids = pd.DataFrame(test_ids) # ID Column

# Merge IDs and Predictions
df_final = pd.concat([df_ids, df_pred], axis=1)

# Save to CSV (No header, No index, Quote none)
df_final.to_csv(VALIDATION_SET_OUTPUT, index=False, header=False, quoting=csv.QUOTE_NONE)

print(f"\n✅ Final submission saved to: {VALIDATION_SET_OUTPUT}")
print(f"Total Rows: {len(df_final)}, Columns: {len(df_final.columns)}")
print("Example Row (Check for Integers):")
print(df_final.iloc[0].values)

--- Starting Final Single Model Prediction ---
Loading BEST Model from: 10879360_task1_best_model.pt
Processing 1053 samples~ (Head+Tail Truncation Strategy)
Applying Class-wise thresholds: [np.float64(0.5400000000000003), np.float64(0.5000000000000002), np.float64(0.5400000000000003), np.float64(0.6000000000000003), np.float64(0.5100000000000002), np.float64(0.5500000000000003), np.float64(0.4300000000000001), np.float64(0.5000000000000002)]

✅ Final submission saved to: ./data/10879360_CW2_task1_validation_results.csv
Total Rows: 1053, Columns: 9
Example Row (Check for Integers):
['9484ac61-0e30-4799-9998-6f74f4cbb204' np.int64(0) np.int64(0)
 np.int64(1) np.int64(0) np.int64(1) np.int64(0) np.int64(0) np.int64(1)]


## End of your code cells

### Evaluation scripts

In [11]:
def read_data(submission_file_path, gold_standard_file_path):
    """
    Read submission and gold standard files.
    Extract student ID from filename.
    """
    # Try to find student ID from the filename (looks for 8 digit numbers)
    id_regex = r'\d{8}'

    user_id = re.findall(id_regex, submission_file_path)
    print("Found your ID: ", user_id)
    if user_id:
        user_id = user_id[0]
    else:
        user_id = 'Unknown'

    # Load submission CSV
    print(f"\nLoading submission file: {submission_file_path}")
    submission_df = pd.read_csv(submission_file_path, sep=',', header=None,
                                quoting=csv.QUOTE_NONE, encoding='utf-8')

    # Load gold standard CSV
    print(f"Loading gold standard file: {gold_standard_file_path}")
    gold_standard_df = pd.read_csv(gold_standard_file_path, header=None)

    # Remove columns 1 and 2 (keep only ID and labels)
    gold_standard_df = gold_standard_df.drop([1, 2], axis=1)
    # Skip header row
    gold_standard_df = gold_standard_df.iloc[1:]

    return submission_df, gold_standard_df, user_id


def match_and_prepare_data(submission_df, gold_standard_df, user_id):
    """
    Match submission rows with gold standard rows by ID.
    Prepare data for evaluation.
    """
    gold_standard_labels = []
    submission_labels = []
    missed_rows = []
    submission_df_copy = submission_df.copy()

    print(f"\nMatching submission with gold standard...")
    print(f"Gold standard rows: {len(gold_standard_df)}")
    print(f"Submission rows: {len(submission_df_copy)}")

    # Match each gold standard row with submission
    for index, row in gold_standard_df.iterrows():
        row = row.reset_index(drop=True)
        row_found = False
        row_id = row[0]

        # Extract gold standard labels
        row_labels = [int(row[i]) for i in range(1, len(row))]
        gold_standard_labels.append(row_labels)

        # Find corresponding submission row
        for sub_index, submission_row in submission_df_copy.iterrows():
            if submission_row[0].strip() == row_id.strip():
                try:
                    # Extract submission labels
                    submission_row_labels = [int(submission_row[i]) for i in range(1, len(submission_row))]
                except:
                    # Handle malformed labels (take first character if multi-digit)
                    submission_row_labels = [int(str(submission_row[i])[0]) for i in range(1, len(submission_row))]

                submission_labels.append(submission_row_labels)
                row_found = True
                submission_df_copy.drop(sub_index, inplace=True)
                break

        if not row_found:
            # If row is missing, add inverse labels (worst possible prediction)
            missed_rows.append(row_id)
            submission_labels.append([0 if label == 1 else 1 for label in row_labels])

    return gold_standard_labels, submission_labels, missed_rows


def evaluate_submission(gold_standard_labels, submission_labels):
    """
    Calculate weighted F1 score.
    """
    print(f"\nCalculating weighted F1 score...")

    # Calculate weighted F1 score (accounts for class imbalance)
    f1_weighted = f1_score(gold_standard_labels, submission_labels, average='weighted')

    return f1_weighted


def print_results(user_id, f1_weighted, missed_rows):
    """
    Print evaluation results to screen.
    """
    print("\n" + "="*70)
    print("YOUR SUBMISSION EVALUATION REPORT")
    print("="*70)

    # Alert if ID not found in filename
    if user_id == 'Unknown':
        print('WARNING: ID not found in filename!')
        print('   Please ensure your filename contains your 8-digit student ID.')
        print()

    print(f"Your ID: {user_id}")
    print()

    # Display F1 score with visual indicator
    print("EVALUATION RESULTS:")
    print(f"   Weighted F1 Score: {f1_weighted:.4f}")
    print()

    # Report missing rows
    if missed_rows:
        print(f"MISSING DATA ({len(missed_rows)} rows not found):")
        print("-" * 70)
        for i, row in enumerate(missed_rows[:10], 1):  # Show first 10
            print(f"    {i}. Row ID: {row}")
        if len(missed_rows) > 10:
            print(f"    ... and {len(missed_rows) - 10} more missing rows")
        print()
        print("TIP: Make sure your submission includes all required rows.")
        print("        Missing rows are penalized with worst possible predictions.")
    else:
        print("DATA COMPLETENESS: All expected rows found in your submission!")

    print()
    print("="*70)
    print()


def evaluate(submission_path, gold_standard_path):
    """
    Main function to run the submission evaluation script.
    """

    submission_file = submission_path
    gold_standard_file = gold_standard_path

    # Check if files exist
    if not os.path.exists(submission_file):
        print(f"Error: Your submission file '{submission_file}' not found!")
        print("Make sure the file path is correct and the file exists.")
        sys.exit(1)

    if not os.path.exists(gold_standard_file):
        print(f"Error: Gold standard file '{gold_standard_file}' not found!")
        print("Make sure you have the correct gold standard file.")
        sys.exit(1)

    try:
        # Step 1: Read data
        submission_df, gold_standard_df, user_id = read_data(submission_file, gold_standard_file)

        # Step 2: Match and prepare data
        gold_standard_labels, submission_labels, missed_rows = match_and_prepare_data(
            submission_df, gold_standard_df, user_id
        )

        # Step 3: Evaluate
        f1_weighted = evaluate_submission(gold_standard_labels, submission_labels)

        # Step 4: Print results
        print_results(user_id, f1_weighted, missed_rows)

    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        print("Please check that your files are in the correct CSV format.")
        print("Each row should contain: ID, label1, label2, label3, ...")
        import traceback
        traceback.print_exc()
        sys.exit(1)

### Evaluate the model on the validation dataset

In [12]:
# Please run the evaluation scripts cell above before running the mark_and_record

# Please make sure that output format is like following (no header row, no title and plot columns):
# 94834c61-0e30-4799-9998-6f74f6sbb204	0	1	0	0	1	0	0	0
# 559sdd28-b6a2-4662-ab55-a6678as26a56	0	0	0	0	0	0	1	0
# b71y3317-04cd-42f5-a380-d21dfasdbd36	0	0	0	0	1	0	0	0

evaluation_results = evaluate(VALIDATION_SET_OUTPUT, VALIDATION_SET)

Found your ID:  ['10879360']

Loading submission file: ./data/10879360_CW2_task1_validation_results.csv
Loading gold standard file: ./data/CW2_validation_dataset.csv

Matching submission with gold standard...
Gold standard rows: 1031
Submission rows: 1031

Calculating weighted F1 score...

YOUR SUBMISSION EVALUATION REPORT
Your ID: 10879360

EVALUATION RESULTS:
   Weighted F1 Score: 0.5845

DATA COMPLETENESS: All expected rows found in your submission!




### Save predictions to formatted file.

In [15]:
# Now please modify the code to format your output csv file.

# Please make sure that output format is like following (no header row, no tilte and plot columns):
# 94834c61-0e30-4799-9998-6f74f6sbb204	0	1	0	0	1	0	0	0
# 559sdd28-b6a2-4662-ab55-a6678as26a56	0	0	0	0	0	0	1	0
# b71y3317-04cd-42f5-a380-d21dfasdbd36	0	0	0	0	1	0	0	0

output_df = df_final  # Replace with your actual DataFrame or output
# For example, if you have a DataFrame named 'output_df', you can save it
assert isinstance(output_df, pd.DataFrame)
assert len(output_df) == NUM_ROWS_TEST, "Output length is not aligned with the testdata.csv."
assert len(output_df.columns) == 9, "Please make sure to follow the format above and keep only IDs and 8 columns of prediction."
output_df.to_csv(f'./{STUDENT_ID}_CW2_task1_results.csv', index=False, header=False)